# Setting Up

As usual, we'll first want to select our Conda environment in the top right, and then we can begin running the analysis one cell at a time using `Ctrl + Enter`.

Today's work directly depends on having downloaded the `diann.zip` file and unzipping its contents into a `diann` subdirectory in the Lesson 3 directory. This can take a bit, so it's well worth getting that download started well before you want to start the analysis proper.

In [ ]:
from collections.abc import Sequence
from pathlib import Path
from warnings import catch_warnings, simplefilter

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn as sns
from scipy.stats import combine_pvalues, false_discovery_control, mannwhitneyu, ttest_ind
from scipy.stats import shapiro as shapiro_wilk

---

In [ ]:
# Here as usual, we're defining paths which we'll use to load in our data
METADATA_FILE = Path("./metadata.tsv")
DATA_BASE_PATH = Path("./diann")

# Here we'll be defining some constants which we'll use to determine "significant" results in our dataset
ALPHA = 0.05
LOG2_FOLD_CHANGE_THRESHOLD = 1

# This is a set of columns that we'll want to include in all of our splits of the ion dataframe
METADATA_COLUMNS = [
    "Protein ID",
    "Protein",
    "Gene",
    "Entry Name",
    "Protein Description",
    "Peptide Sequence",
    "Modified Sequence",
    "Charge",
    "Unique ID",
]

In [ ]:
# This first block is a collection of helper functions for the analysis we'll be doing
# ranging from data loaders to a function which will allow us to split up the dataset
# along the lines of the two tissue types and the control vs. Schizophrenia groups.
def _map_metadata(data: pl.DataFrame, metadata: pl.DataFrame) -> pl.DataFrame:
    """A helper function used to give the raw data columns more human friendly names."""
    annotation = {
        "Protein.Group": "Protein ID",
        "Protein.Ids": "Protein",
        "Genes": "Gene",
        "Protein.Names": "Entry Name",
        "First.Protein.Description": "Protein Description",
        "Stripped.Sequence": "Peptide Sequence",
        "Modified.Sequence": "Modified Sequence",
        "Precursor.Charge": "Charge",
        "Precursor.Id": "Unique ID",
    }
    for row in metadata.iter_rows(named=True):
        annotation[row["FILE"]] = row["CONDITION"]
    return data.rename(annotation)


def load_data(data_path: Path, metadata_path: Path) -> pl.DataFrame:
    """A convenience function to load in the raw ion data from our base data path and automatically apply convenient metadata mapping to the columns."""
    data = pl.read_csv(
        data_path.joinpath("20220120_rklaas_SUN-schizo-cortex-layer1-3+4-6_MBR.pr_matrix.tsv.zst"), separator="\t"
    )
    metadata = pl.read_csv(metadata_path, separator="\t")
    return _map_metadata(data, metadata=metadata)


def split_data(data: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Splits our data by control vs. schizophrenia and which tissue layer it comes from."""
    # The asterisk placed before the list of metadata columns causes python to expand the list into its contents
    return (
        data.select(*METADATA_COLUMNS, (cs.contains("CONTROL") & cs.contains("L1-3"))),
        data.select(*METADATA_COLUMNS, (cs.contains("CONTROL") & cs.contains("L4-6"))),
        data.select(*METADATA_COLUMNS, (cs.contains("SCZ") & cs.contains("L1-3"))),
        data.select(*METADATA_COLUMNS, (cs.contains("SCZ") & cs.contains("L4-6"))),
    )


def filter_ions_for_comparison(control: pl.DataFrame, test: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Filters control and test datasets to only include ions which are present in the other condition."""
    return control.filter(
        pl.col("Unique ID").is_in(
            test.filter(pl.col("Unique ID").is_in(control.select(pl.implode("Unique ID")).to_series()))
            .select(pl.implode("Unique ID"))
            .to_series()
        )
    ), test.filter(
        pl.col("Unique ID").is_in(
            control.filter(pl.col("Unique ID").is_in(test.select(pl.implode("Unique ID")).to_series()))
            .select(pl.implode("Unique ID"))
            .to_series()
        )
    )


# This block consists of functions which we'll use to determine
# if any given column of our data is normally distributed, and
# to then apply the appropriate test
def determine_normality(data: pl.DataFrame) -> pl.DataFrame:
    """Applies the shapiro-wilk test for normality, automatically assuming that the data is not normally distributed if the assumptions of the test cannot be met."""

    def _shapiro_wilk(intensities: Sequence[float]) -> float:
        return float(shapiro_wilk(intensities, nan_policy="omit")[1])

    def _fdr_control(p_values: Sequence[float]) -> float:
        return false_discovery_control(np.nan_to_num(p_values))

    with catch_warnings():
        simplefilter("ignore")
        return (
            data.with_columns(
                cs.contains("Intensity")
                .map_elements(_shapiro_wilk, return_dtype=pl.Float64)
                .alias("Shapiro-Wilk P-value")
            )
            .with_columns(
                pl.col("Shapiro-Wilk P-value")
                .fill_null(0)
                .map_elements(_fdr_control, return_dtype=pl.Float64)
                .alias("Shapiro-Wilk Adj. P-value")
            )
            .with_columns(pl.col("Shapiro-Wilk Adj. P-value").ge(ALPHA).alias("Is Normally Distributed"))
            .drop(cs.contains("Shapiro-Wilk"))
        )


def apply_statistical_test(control: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """Runs the appropriate test, chosen from an indepdent samples ttest or a Mann-Whitney U test, on each ion."""
    with catch_warnings():
        simplefilter("ignore")

        def _ttest(control_intensities: Sequence[float], test_intensities: Sequence[float]) -> float:
            control_intensities = np.array(control_intensities, dtype=np.float64)
            test_intensities = np.array(test_intensities, dtype=np.float64)
            return float(
                ttest_ind(
                    control_intensities[~np.isnan(control_intensities)],
                    test_intensities[~np.isnan(test_intensities)],
                    equal_var=False,
                    nan_policy="omit",
                )[1]
            )

        def _mannwhitneyu(control_intensities: Sequence[float], test_intensities: Sequence[float]) -> float:
            control_intensities = np.array(control_intensities, dtype=np.float64)
            test_intensities = np.array(test_intensities, dtype=np.float64)
            return float(
                mannwhitneyu(
                    control_intensities[~np.isnan(control_intensities)], test_intensities[~np.isnan(test_intensities)]
                )[1]
            )

        # This applies the shapiro-wilk test to determine if each ion is normally distributed
        control = control.with_columns(pl.concat_list(cs.contains("Intensity").alias("Control Intensity"))).select(
            *METADATA_COLUMNS, "Control Intensity"
        )
        control = determine_normality(control)
        test = test.with_columns(pl.concat_list(cs.contains("Intensity").alias("Test Intensity"))).select(
            *METADATA_COLUMNS, "Test Intensity"
        )
        test = determine_normality(test)

        # Here we join our control and test data to facilitate performing the appropriate test
        control = control.join(
            test.select("Unique ID", "Test Intensity", "Is Normally Distributed"),
            on="Unique ID",
            how="inner",
            suffix="_test_condition",
        )

        parametric = control.filter(
            pl.col("Is Normally Distributed").eq(True) & pl.col("Is Normally Distributed_test_condition").eq(True)
        ).drop(cs.contains("Is Normally Distributed"))
        parametric = parametric.with_columns(
            pl.struct("Control Intensity", "Test Intensity")
            .map_elements(lambda x: _ttest(x["Control Intensity"], x["Test Intensity"]), return_dtype=pl.Float64)
            .alias("P-value")
        )

        nonparametric = control.filter(
            pl.col("Is Normally Distributed").eq(False) | pl.col("Is Normally Distributed_test_condition").eq(False)
        ).drop(cs.contains("Is Normally Distributed"))
        nonparametric = nonparametric.with_columns(
            pl.struct("Control Intensity", "Test Intensity")
            .map_elements(lambda x: _mannwhitneyu(x["Control Intensity"], x["Test Intensity"]), return_dtype=pl.Float64)
            .alias("P-value")
        )
    return pl.concat([parametric, nonparametric], how="vertical").fill_nan(1.0).sort("P-value")


# Finally we have a set of functions which we'll use to calculate our final stats for our proteins
def _determine_fold_change(data: pl.DataFrame) -> pl.DataFrame:
    """Determines the average fold change per-protein."""
    return (
        data.group_by("Protein ID")
        .agg(
            pl.col("Control Intensity").explode(empty_as_null=False).mean(),
            pl.col("Test Intensity").explode(empty_as_null=False).mean(),
            "P-value",
        )
        .with_columns(
            pl.col("Test Intensity").fill_null(0).truediv(pl.col("Control Intensity")).fill_null(0).alias("Fold Change")
        )
        .select("Protein ID", "Fold Change", "P-value")
    )


def _deterine_protein_p_val(data: pl.DataFrame) -> pl.DataFrame:
    """Calculates the BH adjusted P-value for each protein from its ions after using Fisher's method."""

    def _combine_p_vals(p_values: Sequence[float]) -> float:
        return float(combine_pvalues(false_discovery_control(np.nan_to_num(p_values, nan=1)))[1])

    return data.with_columns(
        pl.col("P-value").map_elements(_combine_p_vals, return_dtype=pl.Float64).alias("Adj. P-value")
    ).drop("P-value")


def determine_protein_stats(data: pl.DataFrame) -> pl.DataFrame:
    """Calculates BH Adjusted P-value for each protein using Fisher's method, and its corresponding fold change."""
    return (
        _deterine_protein_p_val(_determine_fold_change(data))
        .sort(["Adj. P-value", "Fold Change"], descending=(False, True))
        .with_columns(pl.col("Fold Change").replace([np.inf, -np.inf, 0], 1))
        .with_columns(
            pl.col("Adj. P-value").log10().neg().alias("-Log10 Adj. P-value"),
            pl.col("Fold Change").log(base=2).neg().alias("Log2 Fold Change"),
        )
    )

---

## Exercise 1
Below, we'll be loading in the data for our individual ions, splitting it into our four groups of interest, and then filtering each dataset so it only contains ions present in the dataset.

1. Why do we want to filter our dataset so that each group contains the same ions?
2. How else could we potentially handle one group, either schizophrenia or control, having no datapoints for a particular ion?

In [ ]:
# First, we're just loading our data in, and renaming a few columns to make inspecting it easier
data = load_data(DATA_BASE_PATH, METADATA_FILE)

# Here, we're then unpacking the data into our four groups of interest by disease state and tissue type
controll1_3, controll4_6, scz1_3, scz4_6 = split_data(data)

# Finally, we filter our data for each comparison so that control and scz contain the same ions
controll1_3_filtered, scz1_3_filtered = filter_ions_for_comparison(controll1_3, scz1_3)

display(data)

---

## Exercise 2
Below, we'll determine the statistical test which is appropriate to use for each ion, run it to get a p-value, and then calculate our protein level statistics.

1. How do we check if we should use a parametric or nonparametric test for each ion?
2. Why is it important that we check?
3. Why might we choose to apply multiple hypothesis correction when we're combining p-values for our individual ions from a protein?
4. Where else might be sensible places to apply multiple hypothesis correction?

In [ ]:
# Here we determine the appropriate statistical test for each ion, then run said test
l1_3 = apply_statistical_test(controll1_3_filtered, scz1_3_filtered)
display(l1_3)

In [ ]:
# Here we're calculating what our proteins' fold change and adjusted p-values ought to be
l1_3_proteins = determine_protein_stats(l1_3)
display(l1_3_proteins)

In [ ]:
# Finally, we filter to what hits we would call significant at our predetermined thresholds
significant_hits = l1_3_proteins.filter(
    pl.col("-Log10 Adj. P-value").gt(-np.log10(ALPHA)) & pl.col("Log2 Fold Change").ge(LOG2_FOLD_CHANGE_THRESHOLD)
)
display(significant_hits)

---

## Exercise 3
In the following block, we make what's called a volcano plot. These typically display many data points using their p-values (the chance that an effect is due to chance), and their fold change (how big the difference between conditions is).

Consider the following questions:
1. Why do we use a logarithmic scale for each axis? Try changing to the unscaled versions, and see what happens.
2. What does it mean if a data point passes our significance threshold for fold change but not p-value?
3. What about the opposite scenario?

In [ ]:
plt.rcParams["font.size"] = 14
palette = {"orange": "#fe8019", "green": "#b8bb26"}
fig, ax = plt.subplots(figsize=(6, 6))

plot_df = l1_3_proteins.with_columns(
    color=pl.when(
        pl.col("-Log10 Adj. P-value").gt(-np.log10(ALPHA)) & pl.col("Log2 Fold Change").ge(LOG2_FOLD_CHANGE_THRESHOLD)
    )
    .then(pl.lit("orange"))
    .otherwise(pl.lit("green"))
)

sns.scatterplot(
    plot_df, x="Log2 Fold Change", y="-Log10 Adj. P-value", hue="color", palette=palette, size=1, alpha=1, linewidth=0
)

# Here we show the cutoffs we use to determine significance for our data
ax.axhline(y=-np.log10(ALPHA), color="k", linestyle="--")
for val in [-LOG2_FOLD_CHANGE_THRESHOLD, LOG2_FOLD_CHANGE_THRESHOLD]:
    ax.axvline(x=val, color="k", linestyle="--")

# A useful trick for setting axis boundaries and a sensible set of axis ticks
xlim = np.ceil(plot_df.select(pl.col("Log2 Fold Change").abs().max()).to_series().item())
plt.xlim(-xlim, xlim)
plt.xticks(np.linspace(-xlim, xlim, 7))

# You can format labels with LaTeX if you want to get special formatting in them
plt.xlabel(r"$Log_2\text{ Fold Change}$", fontdict={"fontsize": 16})
plt.ylabel(r"$-Log_{10}\text{ Adj. P-value}$", fontdict={"fontsize": 16})
plt.legend("", frameon=False)
plt.tight_layout()
plt.show()

---

## Challenges
1. Repeat the process from above to determine what proteins (if any) are significant in the layer 4-6 samples when comparing between Control and Schizophrenia.
2. Make a volcano plot of the resulting protein dataframe.